In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-transpiler
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Transpile circuits remotamente com o Qiskit Transpiler Service

> **Danger:** A partir de 18 de julho de 2025, o serviço está sendo migrado para suportar a nova IBM Quantum&reg; Platform e não está disponível. Para passes de IA, você pode usar o [modo local](/guides/ai-transpiler-passes#run-the-ai-transpiler-passes-locally-or-on-the-cloud).
> 
>     O serviço está em versão beta e sujeito a alterações.
>     Se você tiver feedback ou quiser entrar em contato com a equipe de desenvolvimento, use este canal do [Qiskit Slack Workspace](https://qiskit.slack.com/archives/C06KF8YHUAU).

O Qiskit Transpiler Service disponibiliza capacidades de transpilação na nuvem. Além das capacidades do transpilador local do Qiskit, suas tarefas de transpilação podem se beneficiar tanto dos recursos de nuvem do IBM Quantum quanto de passes de transpilação com IA.

O Qiskit Transpiler Service oferece uma biblioteca Python para integrar facilmente esse serviço e suas capacidades nos seus padrões e fluxos de trabalho atuais do Qiskit. Este serviço está disponível apenas para usuários dos planos IBM Quantum Premium Plan, Flex Plan e On-Prem (via IBM Quantum Platform API).

<span id="install-transpiler-service"></span>
## Instale o pacote qiskit-ibm-transpiler
Para usar o Qiskit Transpiler Service, instale o pacote `qiskit-ibm-transpiler`:

In [ ]:
from qiskit.circuit.library import efficient_su2
from qiskit_ibm_transpiler.transpiler_service import TranspilerService

circuit = efficient_su2(101, entanglement="circular", reps=1)

cloud_transpiler_service = TranspilerService(
    backend_name="ibm_torino",
    ai="false",
    optimization_level=3,
)
transpiled_circuit = cloud_transpiler_service.run(circuit)

O pacote autentica automaticamente usando suas [credenciais da IBM Quantum Platform](/guides/cloud-setup), de forma alinhada com a [forma como o Qiskit Runtime as gerencia](/guides/initialize-account):
- Variável de ambiente: `QISKIT_IBM_TOKEN`
- Arquivo de configuração `~/.qiskit/qiskit-ibm.json` (na seção `default-ibm-quantum`).

*Nota*: Este pacote requer o Qiskit SDK v1.X.

## Opções de transpilação do qiskit-ibm-transpiler
- `backend_name` (opcional, str) — O nome de um backend conforme esperado pelo QiskitRuntimeService (por exemplo, `ibm_torino`). Se definido, o método de transpilação usa o layout do backend especificado para a operação de transpilação. Se qualquer outra opção que afete essas configurações for definida, como `coupling_map`, as configurações de `backend_name` serão substituídas.
- `coupling_map` (opcional, List[List[int]]) — Uma lista de coupling map válida (por exemplo, [[0,1],[1,2]]). Se definida, o método de transpilação usa esse coupling map para a operação de transpilação. Quando definido, substitui qualquer valor especificado para `target`.
- `optimization_level` (int) — O nível de otimização potencial a ser aplicado durante o processo de transpilação. Os valores válidos são [1,2,3], onde 1 é a menor otimização (e a mais rápida) e 3 é a maior otimização (e a mais intensiva em tempo).
- `ai` ("true", "false", "auto") — Se as capacidades com IA devem ser usadas durante a transpilação. As capacidades com IA disponíveis podem ser para passes de transpilação `AIRouting` ou outros métodos de síntese com IA. Se esse valor for `"true"`, o serviço aplica diferentes passes de transpilação com IA dependendo do `optimization_level` solicitado. Se for `"false"`, usa os recursos mais recentes de transpilação do Qiskit sem IA. Por fim, se for `"auto"`, o serviço decide se aplica os passes heurísticos padrão do Qiskit ou os passes com IA com base no seu circuit.
- `qiskit_transpile_options` (dict) — Um objeto dicionário Python que pode incluir qualquer outra opção válida no [método `transpile()` do Qiskit](defaults-and-configuration-options). Se o `qiskit_transpile_options` incluir `optimization_level`, ele será descartado em favor do `optimization_level` especificado como parâmetro de entrada. Se o `qiskit_transpile_options` incluir qualquer opção não reconhecida pelo método `transpile()` do Qiskit, a biblioteca levantará um erro.

Para mais informações sobre os métodos disponíveis do `qiskit-ibm-transpiler`, consulte a [referência da API do qiskit-ibm-transpiler](https://docs.quantum.ibm.com/api/qiskit-ibm-transpiler). Para saber mais sobre a API do serviço, consulte a [documentação da REST API do Qiskit Transpiler Service.](https://docs.quantum.ibm.com/api/qiskit-transpiler-service-rest)

## Exemplos
Os exemplos a seguir demonstram como transpillar circuits usando o Qiskit Transpiler Service com diferentes parâmetros.

1. Crie um circuit e chame o Qiskit Transpiler Service para transpilar o circuit com `ibm_torino` como `backend_name`, 3 como `optimization_level`, e sem usar IA durante a transpilação.

In [ ]:
from qiskit.circuit.library import efficient_su2
from qiskit_ibm_transpiler.transpiler_service import TranspilerService

circuit = efficient_su2(101, entanglement="circular", reps=1)

cloud_transpiler_service = TranspilerService(
    backend_name="ibm_torino",
    ai="true",
    optimization_level=1,
)
transpiled_circuit = cloud_transpiler_service.run(circuit)

*Nota*: você só pode usar dispositivos `backend_name` aos quais tem acesso com sua conta IBM Quantum. Além do `backend_name`, o `TranspilerService` também aceita `coupling_map` como parâmetro.

2. Produza um circuit semelhante e transfile-o, solicitando capacidades de transpilação com IA ao definir o sinalizador `ai` como `True`:

In [ ]:
from qiskit.circuit.library import efficient_su2
from qiskit_ibm_transpiler.transpiler_service import TranspilerService

circuit = efficient_su2(101, entanglement="circular", reps=1)

cloud_transpiler_service = TranspilerService(
    backend_name="ibm_torino",
    ai="auto",
    optimization_level=1,
)
transpiled_circuit = cloud_transpiler_service.run(circuit)

3. Produza um circuit semelhante e transfile-o, deixando o serviço decidir se deve usar os passes de transpilação com IA.